# 05 · Automated KPI report and cross-checks

> **All data in this notebook are synthetic** (fictional Indian retail brokerage, 10,000 users, Nov 2025 - Oct 2026, INR, snapshot 2026-10-31). No real company data are used.

Computes every headline KPI, compares it with the validated Phase 3 results, and reconciles Python against the SQLite database.

In [1]:
import sys, warnings; sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from fintech_analytics import config as C, loading, cleaning, eda, funnel, campaign, cohort, segmentation, support, kpi_report
%matplotlib inline
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
d = loading.load_all(); users, act, tix, camp = d['users'], d['activity'], d['tickets'], d['campaigns']
print(f'users={len(users):,}  monthly rows={len(act):,}  tickets={len(tix):,}  campaigns={len(camp)}  (ALL DATA SYNTHETIC)')

users=10,000  monthly rows=64,645  tickets=7,614  campaigns=10  (ALL DATA SYNTHETIC)


In [2]:
k, cmp_, text = kpi_report.build_report(d)
print(f"{(cmp_.status=='MATCH').sum()}/{len(cmp_)} KPIs match Phase 3 within tolerance")
cmp_

31/31 KPIs match Phase 3 within tolerance


,kpi,python_value,phase3_value,abs_diff,tolerance,status
0,total_registrations,10000.0000,10000.0000,0.0,0.000,MATCH
1,mature_users,9116.0000,9116.0000,0.0,0.000,MATCH
2,funnel_registered,9116.0000,9116.0000,0.0,0.000,MATCH
3,funnel_kyc,6429.0000,6429.0000,0.0,0.000,MATCH
4,funnel_bank,5645.0000,5645.0000,0.0,0.000,MATCH
5,funnel_deposit,4340.0000,4340.0000,0.0,0.000,MATCH
6,funnel_trade,3462.0000,3462.0000,0.0,0.000,MATCH
7,conv_kyc_pct,70.5200,70.5200,0.0,0.060,MATCH
8,conv_bank_pct,87.8100,87.8100,0.0,0.060,MATCH
9,conv_deposit_pct,76.8800,76.8800,0.0,0.060,MATCH


## Python vs SQL reconciliation
The same KPIs recomputed with SQL on `fintech_analytics.db` (build it first with `04_sql/build_database.py`).

In [3]:
import sqlite3
con = sqlite3.connect(C.DB_PATH)
sql_funnel = pd.read_sql_query('''SELECT COUNT(*) reg, SUM(kyc_asof) kyc, SUM(bank_asof) bank, SUM(deposit_asof) dep, SUM(trade_asof) trd FROM users WHERE funnel_mature=1''', con).iloc[0].tolist()
py_funnel = [k['funnel_registered'], k['funnel_kyc'], k['funnel_bank'], k['funnel_deposit'], k['funnel_trade']]
sql_m3 = pd.read_sql_query('SELECT 100*AVG(active_in_month) v FROM monthly_activity WHERE months_since_registration=3', con).v[0]
print('funnel  SQL:', sql_funnel, '\nfunnel  PY :', py_funnel, '\nidentical:', sql_funnel == py_funnel)
print(f"M3 retention  SQL {sql_m3:.2f}%  PY {k['m3_retention_pct']:.2f}%  identical: {abs(sql_m3-k['m3_retention_pct'])<1e-9}")

funnel  SQL: [9116, 6429, 5645, 4340, 3462] 
funnel  PY : [9116, 6429, 5645, 4340, 3462] 
identical: True
M3 retention  SQL 56.01%  PY 56.01%  identical: True


## Report text (also saved to `05_python/outputs/KPI_Report.md`)

In [4]:
from IPython.display import Markdown
Markdown(text.split('## Verification detail')[0])

# Automated KPI Report

> **All data are synthetic** (fictional Indian retail brokerage; 10,000 users; Nov 2025 - Oct 2026; INR; snapshot 2026-10-31).
> Generated by `fintech_analytics.kpi_report`. Retention/inactivity flags are proxies, not confirmed churn.

**Verification vs validated Phase 3 results: 31/31 KPIs match within tolerance.**

## Headline KPIs

| KPI | Value |
|---|---|
| Registrations (total / mature cohort) | 10,000 / 9,116 |
| Registered -> first trade (mature) | 38.0% |
| M1 / M3 / M6 retention (calendar-month proxy) | 67.6% / 56.0% / 40.7% |
| MAU, Oct-2026 | 4,611 |
| Dormant 3 months (churn proxy) | 22.0% |
| Repeat traders (share of ever-traders, >= 4 observed months) | 69.3% |
| Support tickets / per 1,000 users | 7,614 / 761 |
| Median / mean resolution time | 9.4 h / 10.9 h |
| CSAT (mean, rated tickets) | 3.83 |
| Unresolved tickets (open + escalated) | 1,371 |

## Funnel (mature cohort)

| stage         |   reached |   pct_of_registered |   step_conversion_pct |   users_lost |
|:--------------|----------:|--------------------:|----------------------:|-------------:|
| Registered    |      9116 |              100    |                nan    |          nan |
| KYC Completed |      6429 |               70.52 |                 70.52 |         2687 |
| Bank Linked   |      5645 |               61.92 |                 87.81 |          784 |
| First Deposit |      4340 |               47.61 |                 76.88 |         1305 |
| First Trade   |      3462 |               37.98 |                 79.77 |          878 |
| Active (30d)  |      2808 |               30.8  |                 81.11 |          654 |

## Campaign scorecard

| campaign_name     | channel      |   registered_users |   cac_inr |   first_trade_rate_mature_pct |   cost_per_first_trader_inr |   m3_retention_pct |
|:------------------|:-------------|-------------------:|----------:|------------------------------:|----------------------------:|-------------------:|
| Customer Referral | Referral     |               1036 |    231.66 |                         52.89 |                      438    |              61.2  |
| Organic Social    | Organic      |                704 |    nan    |                         48.15 |                      nan    |              58.64 |
| Organic Search    | Organic      |               1174 |    nan    |                         46.55 |                      nan    |              61.38 |
| Generic Search    | Paid Search  |               1553 |    392.79 |                         37.36 |                     1051.4  |              54.44 |
| Brand Search      | Paid Search  |               1091 |    384.97 |                         35.47 |                     1085.41 |              54.7  |
| Email Nurture     | CRM          |                561 |    160.43 |                         33.98 |                      472.06 |              55.97 |
| Finance Affiliate | Affiliate    |                757 |    409.51 |                         33.67 |                     1216.23 |              54.35 |
| Campus Events     | Partnerships |                580 |    327.59 |                         31.14 |                     1051.83 |              50.77 |
| Video Awareness   | Paid Social  |               1168 |    385.27 |                         30.78 |                     1251.84 |              53.1  |
| Education Social  | Paid Social  |               1376 |    276.16 |                         29.99 |                      920.78 |              54.85 |

## Journey segments

| journey_segment                                          |   users |   avg_tenure_days |   avg_trading_value_6m_inr |   tickets_per_user |   share_pct |
|:---------------------------------------------------------|--------:|------------------:|---------------------------:|-------------------:|------------:|
| 01 New / in-flight onboarding                            |     601 |             11.12 |                    6351.05 |               0.6  |        6.01 |
| 02 KYC-pending (stalled)                                 |    2831 |            187.31 |                    4408.67 |               1    |       28.31 |
| 03 Bank-link pending (stalled)                           |     818 |            194.92 |                    4132.22 |               0.83 |        8.18 |
| 04 Deposit-pending (stalled)                             |    1333 |            194.95 |                    4491.46 |               0.78 |       13.33 |
| 05 Deposited, never traded (stalled)                     |     881 |            196.56 |                    3909.27 |               0.72 |        8.81 |
| 06 First-time trader (<=30d since first trade)           |     317 |             38.93 |                   17074.4  |               0.59 |        3.17 |
| 07 Repeat trader (2-3 trading months of last 6)          |    1684 |            200.77 |                   36730.1  |               0.58 |       16.84 |
| 08 High-engagement trader (>=4 trading months of last 6) |     896 |            210.12 |                   61159.9  |               0.6  |        8.96 |
| 09 Inactive trader (no activity Aug-Oct)                 |     352 |            268.88 |                   21908.3  |               0.59 |        3.52 |
| 10 Occasional trader (0-1 trading months of last 6)      |     287 |            184.02 |                   13700.6  |               0.63 |        2.87 |

## Support categories

| category               |   tickets |   share_pct |   median_resolution_h |   avg_csat |   unresolved |
|:-----------------------|----------:|------------:|----------------------:|-----------:|-------------:|
| KYC & verification     |      1727 |       22.68 |                  9.5  |       3.77 |          316 |
| Trading & orders       |      1313 |       17.24 |                  9.3  |       3.85 |          226 |
| Bank linking           |      1246 |       16.36 |                  9.6  |       3.81 |          242 |
| Deposits & withdrawals |      1198 |       15.73 |                  9.55 |       3.83 |          218 |
| App navigation         |       892 |       11.72 |                  9.7  |       3.86 |          165 |
| Account access         |       732 |        9.61 |                  9.1  |       3.89 |          118 |
| Fees & charges         |       506 |        6.65 |                  9.05 |       3.83 |           86 |

